# CRUD Operations in ChromaDB (Adding and Querying Documents)

**Level:** Scratch → Intermediate
**Runs on:** Google Colab (Free CPU)

**CRUD** = **C**reate, **R**ead, **U**pdate, **D**elete — the four basic operations every database supports.

### What you will learn
1. **Create** — set up a client and collection
2. **Create (Add)** — insert documents with ids and metadata
3. **Read (Query)** — semantic search + get-by-id + filters
4. **Update** — modify existing documents/metadata
5. **Delete** — remove documents by id or by filter
6. Checking collection state (`count`, `peek`, `list_collections`)

### 1. Install ChromaDB
Run this once.

In [17]:
%pip install chromadb

In [18]:
!pip install -q chromadb

### 2. CREATE — Client and Collection

In [20]:
import chromadb

# In-memory client (resets when runtime restarts) - great for learning
client = chromadb.Client()

# Get or create a fresh collection to practice CRUD operations on safely
collection = client.get_or_create_collection(name="crud_demo")

print("Collection created or retrieved:", collection.name)
print("Current document count:", collection.count())

Collection created or retrieved: crud_demo
Current document count: 2


### 3. CREATE — Adding Documents (`add`)

Each document needs a unique `id`. `metadata` and `documents` are optional extras that describe each entry.

In [21]:
collection.add(
    documents=[
        "Python is a popular programming language for AI.",
        "The weather today is sunny and warm.",
        "Machine learning models learn patterns from data.",
        "I enjoy playing football on weekends.",
        "Deep learning uses neural networks with many layers."
    ],
    metadatas=[
        {"category": "tech"},
        {"category": "weather"},
        {"category": "tech"},
        {"category": "sports"},
        {"category": "tech"}
    ],
    ids=["doc1", "doc2", "doc3", "doc4", "doc5"]
)

print("Documents added! Total count:", collection.count())

Documents added! Total count: 6


**Discussion Point:** If you try to `add()` an `id` that already exists, ChromaDB will raise an error. Use `upsert()` (shown later) if you want "add or update" behavior instead.

### 4. READ — Peek at the Collection

`peek()` shows a quick sample of what's inside, without needing a query.

In [22]:
preview = collection.peek(limit=3)

for doc_id, doc in zip(preview["ids"], preview["documents"]):
    print(doc_id, "->", doc)

doc6 -> Introduction to Generative AI
doc1 -> Updated Python Basics
doc2 -> The weather today is sunny and warm.


### 5. READ — Get Documents by ID
Use `get()` when you know exactly which id(s) you want — this is an exact lookup, not a similarity search.

In [23]:
result = collection.get(ids=["doc1", "doc3"])

for doc_id, doc, meta in zip(result["ids"], result["documents"], result["metadatas"]):
    print(f"{doc_id} | {meta} | {doc}")

doc1 | {'category': 'programming'} | Updated Python Basics
doc3 | {'category': 'tech'} | Machine learning models learn patterns from data.


### 6. READ — Semantic Search (`query`)

This is the core "vector database" feature: search by **meaning**, not exact words.

In [24]:
results = collection.query(
    query_texts=["Tell me about AI and neural networks"],
    n_results=2
)

for doc_id, doc, distance in zip(results["ids"][0], results["documents"][0], results["distances"][0]):
    print(f"{doc_id} | distance={distance:.4f} | {doc}")

doc5 | distance=0.9317 | Deep learning uses neural networks with many layers.
doc6 | distance=1.1315 | Introduction to Generative AI


### 7. READ — Filtering with `where` (Metadata Filter)

Combine semantic search with exact filters — e.g., "only search within the 'tech' category".

In [25]:
results = collection.query(
    query_texts=["something fun to do outside"],
    n_results=3,
    where={"category": "sports"}
)

for doc_id, doc, distance in zip(results["ids"][0], results["documents"][0], results["distances"][0]):
    print(f"{doc_id} | distance={distance:.4f} | {doc}")

doc4 | distance=1.3895 | I enjoy playing football on weekends.


### 8. UPDATE — Modifying Existing Documents

`update()` changes the text and/or metadata of documents that **already exist** (matched by `id`).

In [26]:
collection.update(
    ids=["doc4"],
    documents=["I enjoy playing football and basketball on weekends."],
    metadatas=[{"category": "sports", "updated": True}]
)

updated = collection.get(ids=["doc4"])
print(updated["documents"])
print(updated["metadatas"])

['I enjoy playing football and basketball on weekends.']
[{'category': 'sports', 'updated': True}]


### 9. UPSERT — Add or Update in One Step

`upsert()` is useful when you're not sure if the `id` already exists: if it exists, it updates it; if not, it creates it.

In [27]:
collection.upsert(
    documents=[
        "Deep learning uses neural networks with many layers and GPUs.",  # existing id -> will update
        "The Eiffel Tower is located in Paris."                           # new id -> will create
    ],
    ids=["doc5", "doc6"]
)

print("Total count after upsert:", collection.count())
print(collection.get(ids=["doc5", "doc6"])["documents"])

Total count after upsert: 6
['The Eiffel Tower is located in Paris.', 'Deep learning uses neural networks with many layers and GPUs.']


### 10. DELETE — Removing Documents by ID

In [28]:
collection.delete(ids=["doc2"])

print("Total count after delete:", collection.count())
print("Remaining ids:", collection.get()["ids"])

Total count after delete: 5
Remaining ids: ['doc6', 'doc1', 'doc3', 'doc4', 'doc5']


### 11. DELETE — Removing Documents by Filter

You can also delete every document that matches a metadata filter, without listing ids manually.

In [29]:
collection.delete(where={"category": "tech"})

print("Total count after category delete:", collection.count())
print("Remaining ids:", collection.get()["ids"])

Total count after category delete: 3
Remaining ids: ['doc6', 'doc1', 'doc4']


### 12. Verifying Everything (Final State Check)

In [30]:
final_state = collection.get()

print("Final document count:", collection.count())
for doc_id, doc, meta in zip(final_state["ids"], final_state["documents"], final_state["metadatas"]):
    print(f"{doc_id} | {meta} | {doc}")

Final document count: 3
doc6 | {'category': 'genai'} | The Eiffel Tower is located in Paris.
doc1 | {'category': 'programming'} | Updated Python Basics
doc4 | {'updated': True, 'category': 'sports'} | I enjoy playing football and basketball on weekends.


### 13. Key Takeaways (Recap for Students)

| Operation | Method | Notes |
|---|---|---|
| Create | `client.create_collection()` | Makes a new collection (errors if it already exists) |
| Create/Add | `collection.add()` | Insert new documents; errors on duplicate `id` |
| Read (exact) | `collection.get()` | Fetch by `id`, no similarity involved |
| Read (semantic) | `collection.query()` | Search by meaning, returns closest matches + distances |
| Read (preview) | `collection.peek()` | Quick look at a few stored documents |
| Update | `collection.update()` | Modify existing document/metadata by `id` |
| Add or Update | `collection.upsert()` | Creates if missing, updates if it exists |
| Delete | `collection.delete()` | Remove by `id` list or by `where` metadata filter |

### 14. Practice Exercises
1. Add 5 new documents, then use `upsert()` to update 2 of them and create 1 brand-new one in a single call.
2. Delete all documents where `category` equals a value of your choice, then confirm with `collection.count()`.
3. Write a small function `safe_add(collection, id, text)` that uses `upsert()` internally so it never raises a duplicate-id error.

In [31]:
# Step 1: Add 5 new documents

collection.add(
    ids=["doc1", "doc2", "doc3", "doc4", "doc5"],
    documents=[
        "Python Basics",
        "Machine Learning Fundamentals",
        "Deep Learning Introduction",
        "Natural Language Processing",
        "Computer Vision Basics"
    ],
    metadatas=[
        {"category": "programming"},
        {"category": "ai"},
        {"category": "ai"},
        {"category": "ai"},
        {"category": "ai"}
    ]
)

print("Initial 5 documents added.")

# Step 2: Update doc2 and doc4, and create a new doc6

collection.upsert(
    ids=["doc2", "doc4", "doc6"],
    documents=[
        "Updated Machine Learning Fundamentals",
        "Updated Natural Language Processing",
        "Generative AI Overview"
    ],
    metadatas=[
        {"category": "ai"},
        {"category": "ai"},
        {"category": "genai"}
    ]
)

print("2 documents updated and 1 new document created.")

# Verify

results = collection.get(ids=["doc2", "doc4", "doc6"])
print(results)

Initial 5 documents added.
2 documents updated and 1 new document created.
{'ids': ['doc6', 'doc4', 'doc2'], 'embeddings': None, 'documents': ['Generative AI Overview', 'Updated Natural Language Processing', 'Updated Machine Learning Fundamentals'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'category': 'genai'}, {'category': 'ai', 'updated': True}, {'category': 'ai'}]}


In [32]:
# Count before deletion
print("Count before deletion:", collection.count())

# Delete all documents where category = "ai"
collection.delete(
    where={"category": "ai"}
)

print("Documents with category 'ai' deleted.")

# Confirm deletion
print("Count after deletion:", collection.count())

Count before deletion: 6
Documents with category 'ai' deleted.
Count after deletion: 2


In [33]:
def safe_add(collection, doc_id, text):
    collection.upsert(
        ids=[doc_id],
        documents=[text]
    )
    print(f"Document '{doc_id}' added or updated successfully.")


# Example Usage

safe_add(collection, "doc1", "Updated Python Basics")
safe_add(collection, "doc6", "Introduction to Generative AI")

# Verify the documents
print(collection.get(ids=["doc1", "doc6"]))

Document 'doc1' added or updated successfully.
Document 'doc6' added or updated successfully.
{'ids': ['doc6', 'doc1'], 'embeddings': None, 'documents': ['Introduction to Generative AI', 'Updated Python Basics'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'category': 'genai'}, {'category': 'programming'}]}
